# HC3 analog experiment — does the validation-signal finding generalize beyond M4?

**The M4 finding**: checkpoint selection on a genuinely out-of-distribution signal
(reddit+dolly) significantly improved generalization over selection on an easy,
in-distribution signal (reddit, train generators) — confirmed via 5-seed variance,
paired McNemar (p≈0 on all three tests), and bootstrap CIs.

**HC3 has no generator axis** (single generator, ChatGPT) — so the analogous "hard"
signal has to come from the domain axis alone: a domain that's neither used in
training nor reserved as the final test domain. If HC3 has 5 source domains (as the
original paper's Table 2 described: reddit_eli5, finance, open_qa, wiki_csai, medicine),
the design is:

- **Train**: 3 largest domains
- **Easy val** (original): in-domain, disjoint samples from the train domains
- **Hard val** (new): the 4th domain — genuinely unseen during training, but NOT the
  final test domain
- **Test**: the smallest domain — untouched by anything above, final evaluation only

This mirrors the M4 design exactly (easy in-distribution val vs. hard out-of-distribution
val, evaluated on a completely separate held-out set) using the only axis of shift HC3
actually has. If domain counts don't support this 4-way split, the diagnostic below
will say so explicitly rather than silently degrading.

## 1. Setup + checkpoint infrastructure

In [ ]:
!pip install -q transformers "datasets<4.0.0" accelerate scikit-learn pandas numpy nltk tqdm matplotlib shap statsmodels

import os, re, json, random, warnings, math, pickle, shutil, glob
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter
from tqdm.auto import tqdm

import nltk
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True); nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk import word_tokenize, sent_tokenize, pos_tag
from nltk.corpus import stopwords

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, GPT2LMHeadModel, get_cosine_schedule_with_warmup
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from statsmodels.stats.contingency_tables import mcnemar

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

CHECKPOINT_DIR = '/kaggle/working/checkpoints_hc3'
ARTIFACTS_DIR = '/kaggle/working/artifacts_hc3'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

restored = 0
for candidate in glob.glob('/kaggle/input/*/checkpoints_hc3') + glob.glob('/kaggle/input/*/*/checkpoints_hc3'):
    if os.path.isdir(candidate) and os.path.abspath(candidate) != os.path.abspath(CHECKPOINT_DIR):
        for item in os.listdir(candidate):
            dst = os.path.join(CHECKPOINT_DIR, item)
            if not os.path.exists(dst):
                src_path = os.path.join(candidate, item)
                (shutil.copytree if os.path.isdir(src_path) else shutil.copy2)(src_path, dst)
                restored += 1
        break
print(f'Restored {restored} item(s) from a previous session (0 is normal on a fresh run).')

def ckpt_path(name): return os.path.join(CHECKPOINT_DIR, name + '.pkl')
def ckpt_exists(name): return os.path.exists(ckpt_path(name))
def ckpt_save(name, obj):
    tmp = ckpt_path(name) + '.tmp'
    with open(tmp, 'wb') as f: pickle.dump(obj, f)
    os.replace(tmp, ckpt_path(name))
def ckpt_load(name):
    with open(ckpt_path(name), 'rb') as f: return pickle.load(f)

STOPWORDS = set(stopwords.words('english'))
FUNCTION_WORDS = ['the', 'of', 'and', 'to', 'in', 'is', 'that', 'it']
SEEDS = [42, 43, 44, 45, 46]
MIN_WORDS = 30
def _wc(t): return len(t.split())
def clean(t): return re.sub(r'\s+', ' ', str(t)).strip()


## 2. Load HC3 — same verified 3-tier fallback as before

In [ ]:
if ckpt_exists('hc3_raw_dataframe'):
    df_hc3 = ckpt_load('hc3_raw_dataframe')
    print('Loaded HC3 from checkpoint.')
else:
    import requests

    def _load_via_script():
        from datasets import load_dataset
        return load_dataset('Hello-SimpleAI/HC3', 'all', trust_remote_code=True)['train'].to_pandas()

    def _load_via_parquet_revision():
        from datasets import load_dataset
        return load_dataset('Hello-SimpleAI/HC3', 'all', revision='refs/convert/parquet')['train'].to_pandas()

    def _load_via_datasets_server():
        api_url = 'https://datasets-server.huggingface.co/parquet?dataset=Hello-SimpleAI/HC3'
        resp = requests.get(api_url, timeout=30)
        resp.raise_for_status()
        info = resp.json()
        parquet_files = [f for f in info['parquet_files'] if f['config'] == 'all' and f['split'] == 'train']
        if not parquet_files:
            raise RuntimeError('No matching parquet files found via datasets-server API')
        dfs = [pd.read_parquet(f['url']) for f in parquet_files]
        return pd.concat(dfs, ignore_index=True)

    hc3_raw = None
    for loader_fn, desc in [(_load_via_script, 'script-based (trust_remote_code)'),
                             (_load_via_parquet_revision, 'refs/convert/parquet revision'),
                             (_load_via_datasets_server, 'datasets-server API direct parquet')]:
        try:
            hc3_raw = loader_fn()
            print(f'HC3 loaded successfully via: {desc}')
            break
        except Exception as e:
            print(f'  {desc} failed: {e}')
    if hc3_raw is None:
        raise RuntimeError('All three HC3 loading strategies failed -- check internet connectivity / HF Hub status.')

    def explode_hc3(row):
        out = []
        for h in row['human_answers']:
            h = clean(str(h))
            if _wc(h) >= MIN_WORDS: out.append({'text': h, 'domain': row['source'], 'label': 0})
        for a in row['chatgpt_answers']:
            a = clean(str(a))
            if _wc(a) >= MIN_WORDS: out.append({'text': a, 'domain': row['source'], 'label': 1})
        return out

    hc3_rows = []
    for _, row in tqdm(hc3_raw.iterrows(), total=len(hc3_raw), desc='Exploding HC3'):
        hc3_rows.extend(explode_hc3(row))
    df_hc3 = pd.DataFrame(hc3_rows).drop_duplicates(subset=['domain','text']).reset_index(drop=True)
    ckpt_save('hc3_raw_dataframe', df_hc3)

print('HC3 total rows:', len(df_hc3))
domain_sizes = df_hc3.groupby('domain').size().sort_values(ascending=False)
print(domain_sizes)


## 3. Build the 4-way split — train / easy-val / hard-val / test

Requires at least 5 distinct domains with sufficient samples each. If HC3's actual
domain structure doesn't support this, the assertion below fails with a clear message
rather than silently falling back to a weaker design.

In [ ]:
domain_label_counts = df_hc3.groupby(['domain','label']).size().unstack(fill_value=0)
print('Per-domain, per-label counts:')
print(domain_label_counts)

MIN_PER_CLASS = 300   # minimum viable samples per class for any split role
viable_domains = domain_label_counts[(domain_label_counts.get(0,0) >= MIN_PER_CLASS) &
                                      (domain_label_counts.get(1,0) >= MIN_PER_CLASS)]
viable_sorted = viable_domains.sum(axis=1).sort_values(ascending=False)
print()
print('Viable domains (>=300 samples/class), sorted by size:')
print(viable_sorted)

assert len(viable_sorted) >= 5, (
    f'HC3 analog experiment requires >=5 viable domains (3 train + 1 hard-val + 1 test), '
    f'found {len(viable_sorted)}. Cannot proceed with the 4-way design as planned -- '
    f'either lower MIN_PER_CLASS (reduces test/val rigor) or use a 3-way design '
    f'(train + easy-val only, no hard-val comparison) instead.'
)

domains_ranked = list(viable_sorted.index)
HC3_TRAIN_DOMAINS = domains_ranked[:3]
HC3_HARD_VAL_DOMAIN = domains_ranked[3]
HC3_TEST_DOMAIN = domains_ranked[-1]   # smallest viable domain, final test only

assert HC3_HARD_VAL_DOMAIN != HC3_TEST_DOMAIN, 'hard-val and test domain collided -- need >=5 distinct viable domains'
assert HC3_HARD_VAL_DOMAIN not in HC3_TRAIN_DOMAINS

print()
print('Train domains:', HC3_TRAIN_DOMAINS)
print('Hard-val domain (unseen during training, NOT the final test):', HC3_HARD_VAL_DOMAIN)
print('Test domain (final, completely held out):', HC3_TEST_DOMAIN)


In [ ]:
HC3_SEED = 42
N_HC3_TRAIN, N_HC3_EASYVAL, N_HC3_HARDVAL, N_HC3_TEST = 4000, 500, 400, 300

train_pool = df_hc3[df_hc3.domain.isin(HC3_TRAIN_DOMAINS)]
hardval_pool = df_hc3[df_hc3.domain == HC3_HARD_VAL_DOMAIN]
test_pool = df_hc3[df_hc3.domain == HC3_TEST_DOMAIN]

n_train = min(N_HC3_TRAIN, train_pool[train_pool.label==0].shape[0], train_pool[train_pool.label==1].shape[0])
human_shuf = train_pool[train_pool.label==0].sample(frac=1.0, random_state=HC3_SEED).reset_index(drop=True)
train_human = human_shuf.iloc[:n_train]
easyval_human_pool = human_shuf.iloc[n_train:]

ai_shuf = train_pool[train_pool.label==1].sample(frac=1.0, random_state=HC3_SEED).reset_index(drop=True)
train_ai = ai_shuf.iloc[:n_train]
easyval_ai_pool = ai_shuf.iloc[n_train:]

df_train = pd.concat([train_human, train_ai], ignore_index=True).sample(frac=1, random_state=HC3_SEED).reset_index(drop=True)

n_easyval = min(N_HC3_EASYVAL, len(easyval_human_pool), len(easyval_ai_pool))
assert n_easyval >= 100, f'Not enough spare in-domain samples for easy-val: {n_easyval}'
df_val_easy = pd.concat([easyval_human_pool.iloc[:n_easyval], easyval_ai_pool.iloc[:n_easyval]], ignore_index=True)

n_hardval = min(N_HC3_HARDVAL, hardval_pool[hardval_pool.label==0].shape[0], hardval_pool[hardval_pool.label==1].shape[0])
hardval_human = hardval_pool[hardval_pool.label==0].sample(n=n_hardval, random_state=HC3_SEED)
hardval_ai = hardval_pool[hardval_pool.label==1].sample(n=n_hardval, random_state=HC3_SEED)
df_val_hard = pd.concat([hardval_human, hardval_ai], ignore_index=True)

n_test = min(N_HC3_TEST, test_pool[test_pool.label==0].shape[0], test_pool[test_pool.label==1].shape[0])
test_human = test_pool[test_pool.label==0].sample(n=n_test, random_state=HC3_SEED)
test_ai = test_pool[test_pool.label==1].sample(n=n_test, random_state=HC3_SEED)
df_test = pd.concat([test_human, test_ai], ignore_index=True)

all_splits_check = {'train': df_train, 'val_easy': df_val_easy, 'val_hard': df_val_hard, 'test': df_test}
seen = {}
for name, d in all_splits_check.items():
    for t in d['text']:
        assert not (t in seen and seen[t] != name), f'LEAKAGE: text appears in both {seen[t]} and {name}'
        seen[t] = name
print('No text-level leakage across train/val_easy/val_hard/test.')
for name, d in all_splits_check.items():
    print(f'{name:10s}: n={len(d)}, labels={d.label.value_counts().to_dict()}, domains={sorted(d.domain.unique())}')

y_train, y_val_easy, y_val_hard, y_test = df_train.label.values, df_val_easy.label.values, df_val_hard.label.values, df_test.label.values


## 4. Stylometric features — same consistent, self-contained pipeline as Stage-4-v2

Fit once on `df_train`, applied identically to all four splits. No cross-pipeline
reproduction attempted anywhere (that's what caused the M4 scale bug).

In [ ]:
def safe_div(a,b): return a/b if b else 0.0
def lexical_features(tokens):
    n = len(tokens)
    if n == 0: return {'hapax_ratio':0.,'yules_k':0.,'ttr':0.,'avg_word_len':0.}
    freqs = Counter(tokens); V = len(freqs)
    hapax = sum(1 for w,c in freqs.items() if c==1)
    freq_of_freq = Counter(freqs.values())
    sum_i2fi = sum((i**2)*fi for i,fi in freq_of_freq.items())
    return {'hapax_ratio':hapax/n,'yules_k':1e4*(sum_i2fi-n)/(n**2),'ttr':V/n,
            'avg_word_len':float(np.mean([len(w) for w in tokens]))}
def syntactic_features(sentences):
    lens = [len(word_tokenize(s)) for s in sentences] if sentences else [0]
    mean_len, var_len = float(np.mean(lens)), float(np.var(lens))
    burstiness = (var_len-mean_len)/(var_len+mean_len) if (var_len+mean_len)>0 else 0.
    full_text = ' '.join(sentences); n_chars = max(len(full_text),1)
    n_punct = sum(1 for c in full_text if c in '.,;:!?')
    return {'sent_len_variance':var_len,'burstiness':burstiness,'punct_density':n_punct/n_chars,'avg_sent_len':mean_len}
def pos_bigrams(tagged):
    tags = [t for _,t in tagged]; return list(zip(tags, tags[1:]))
def fit_pos_bigram_vocab(train_texts, top_k=36):
    counter = Counter()
    for text in tqdm(train_texts, desc='Fitting POS-bigram vocab (train only)'):
        counter.update(pos_bigrams(pos_tag(word_tokenize(text))))
    return [bg for bg,_ in counter.most_common(top_k)]
def grammatical_features(tagged, vocab):
    bigrams = pos_bigrams(tagged); total = len(bigrams); counts = Counter(bigrams)
    return {f'pos_{a}_{b}': (counts.get((a,b),0)/total if total>0 else 0.) for a,b in vocab}
def build_burrows_reference(train_human_texts, top_words):
    rates = {w: [] for w in top_words}
    for text in train_human_texts:
        tokens = [t.lower() for t in word_tokenize(text)]; n = max(len(tokens),1); freqs = Counter(tokens)
        for w in top_words: rates[w].append(freqs.get(w,0)/n)
    return ({w: float(np.mean(v)) for w,v in rates.items()}, {w: (float(np.std(v)) if np.std(v)>0 else 1.0) for w,v in rates.items()})
def burrows_delta(tokens, ref_mean, ref_std, top_words):
    n = max(len(tokens),1); freqs = Counter(tokens)
    diffs = [abs(((freqs.get(w,0)/n)-ref_mean.get(w,0.))/ref_std.get(w,1.)) for w in top_words]
    return float(np.mean(diffs)) if diffs else 0.
def function_word_ratios(tokens):
    n = max(len(tokens),1); freqs = Counter(t.lower() for t in tokens)
    return {f'func_{w}': freqs.get(w,0)/n for w in FUNCTION_WORDS}

class StylometricExtractor:
    def fit(self, train_texts, train_human_texts, n_bigrams=36, n_burrows_words=20):
        self.bigram_vocab = fit_pos_bigram_vocab(train_texts, n_bigrams)
        all_tok = [w.lower() for t in train_human_texts for w in word_tokenize(t)]
        self.burrows_words = [w for w,_ in Counter(all_tok).most_common(n_burrows_words) if w.isalpha()]
        self.ref_mean, self.ref_std = build_burrows_reference(train_human_texts, self.burrows_words)
        return self
    def transform(self, text, gpt2_ppl_fn=None):
        tokens, sentences = word_tokenize(text), sent_tokenize(text)
        tagged = pos_tag(tokens)
        feats = {}
        feats.update(lexical_features([t.lower() for t in tokens]))
        feats.update(syntactic_features(sentences))
        feats.update(grammatical_features(tagged, self.bigram_vocab))
        feats['burrows_delta'] = burrows_delta([t.lower() for t in tokens], self.ref_mean, self.ref_std, self.burrows_words)
        feats['gpt2_perplexity'] = gpt2_ppl_fn(text) if gpt2_ppl_fn else np.nan
        feats.update(function_word_ratios(tokens))
        return feats
    def transform_batch(self, texts, gpt2_ppl_fn=None, desc='Extracting'):
        rows = [self.transform(t, gpt2_ppl_fn) for t in tqdm(texts, desc=desc)]
        return pd.DataFrame(rows)

CAT_COLS = {
    'lexical': ['hapax_ratio','yules_k','ttr','avg_word_len'],
    'syntactic': ['sent_len_variance','burstiness','punct_density','avg_sent_len'],
}
_gpt2_tok = AutoTokenizer.from_pretrained('gpt2')
_gpt2_lm = GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE).eval()

@torch.no_grad()
def gpt2_perplexity(text, max_len=512):
    ids = _gpt2_tok(text, return_tensors='pt', truncation=True, max_length=max_len).input_ids.to(DEVICE)
    if ids.shape[1] < 2: return float('nan')
    loss = _gpt2_lm(ids, labels=ids).loss
    return float(torch.exp(loss).item())

extractor_ckpt = 'hc3_stylometric_extractor'
if ckpt_exists(extractor_ckpt):
    extractor = ckpt_load(extractor_ckpt)
else:
    extractor = StylometricExtractor().fit(df_train['text'].tolist(), df_train[df_train.label==0]['text'].tolist())
    ckpt_save(extractor_ckpt, extractor)

CAT_COLS['grammatical'] = [f'pos_{a}_{b}' for a,b in extractor.bigram_vocab]
CAT_COLS['authorship'] = ['burrows_delta', 'gpt2_perplexity'] + [f'func_{w}' for w in FUNCTION_WORDS]
feature_columns = CAT_COLS['lexical'] + CAT_COLS['syntactic'] + CAT_COLS['grammatical'] + CAT_COLS['authorship']
assert len(feature_columns) == 54
print('Feature dimension:', len(feature_columns))


In [ ]:
splits_hc3 = {'train': df_train, 'val_easy': df_val_easy, 'val_hard': df_val_hard, 'test': df_test}
X_style = {}
for name, d in splits_hc3.items():
    ckpt_name = f'hc3_style_{name}'
    if ckpt_exists(ckpt_name):
        X_style[name] = ckpt_load(ckpt_name)
    else:
        X_style[name] = extractor.transform_batch(d['text'].tolist(), gpt2_perplexity, f'style: {name}')
        ckpt_save(ckpt_name, X_style[name])

ppl_median = X_style['train']['gpt2_perplexity'].median()
for k in X_style: X_style[k]['gpt2_perplexity'] = X_style[k]['gpt2_perplexity'].fillna(ppl_median)

style_scaler = StandardScaler().fit(X_style['train'][feature_columns].values)
Xs = {k: style_scaler.transform(v[feature_columns].values) for k, v in X_style.items()}
col_index = {c: i for i, c in enumerate(feature_columns)}
def split_by_cat(arr): return {cat: arr[:, [col_index[c] for c in cols]] for cat, cols in CAT_COLS.items()}
Xcat = {k: split_by_cat(v) for k, v in Xs.items()}
category_dims = {k: len(v) for k, v in CAT_COLS.items()}

print('Feature ranges (sanity check -- should all be similarly standardized):')
for name, arr in Xs.items():
    print(f'  {name:10s}: shape={arr.shape}, range=[{arr.min():.2f}, {arr.max():.2f}]')


## 5. Frozen BERT embeddings

In [ ]:
_bert_tok = AutoTokenizer.from_pretrained('distilbert-base-uncased')
_bert_model = AutoModel.from_pretrained('distilbert-base-uncased').to(DEVICE).eval()

@torch.no_grad()
def get_cls_embeddings(texts, batch_size=32, max_length=512, desc='BERT embeddings'):
    embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = texts[i:i+batch_size]
        enc = _bert_tok(batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt').to(DEVICE)
        out = _bert_model(**enc).last_hidden_state[:, 0, :]
        embs.append(out.cpu().numpy())
    return np.vstack(embs)

Xb = {}
for name, d in splits_hc3.items():
    ckpt_name = f'hc3_bert_{name}'
    if ckpt_exists(ckpt_name):
        Xb[name] = ckpt_load(ckpt_name)
    else:
        Xb[name] = get_cls_embeddings(d['text'].tolist(), desc=f'BERT: {name}')
        ckpt_save(ckpt_name, Xb[name])
print('BERT embedding shapes:', {k: v.shape for k, v in Xb.items()})


## 6. Attention-gated model + trainer — identical to the M4 version

In [ ]:
class AttentionGatedHybridDetector(nn.Module):
    def __init__(self, category_dims, bert_dim=768, cat_hidden=32, ffn_out=64):
        super().__init__()
        self.category_names = list(category_dims.keys())
        self.category_encoders = nn.ModuleDict({
            name: nn.Sequential(nn.Linear(dim, cat_hidden), nn.ReLU(), nn.Dropout(0.2))
            for name, dim in category_dims.items()})
        n_cat = len(category_dims)
        self.gate = nn.Linear(cat_hidden * n_cat, n_cat)
        self.style_proj = nn.Sequential(nn.Linear(cat_hidden, ffn_out), nn.ReLU())
        self.classifier = nn.Sequential(
            nn.Linear(bert_dim + ffn_out, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 2))
    def forward(self, bert_emb, category_feats, return_gate=False):
        encoded = [self.category_encoders[name](category_feats[name]) for name in self.category_names]
        concat_encoded = torch.cat(encoded, dim=1)
        gate_weights = torch.softmax(self.gate(concat_encoded), dim=1)
        stacked = torch.stack(encoded, dim=1)
        weighted = (stacked * gate_weights.unsqueeze(-1)).sum(dim=1)
        style_repr = self.style_proj(weighted)
        logits = self.classifier(torch.cat([bert_emb, style_repr], dim=1))
        return (logits, gate_weights) if return_gate else logits

class CategoryDataset(Dataset):
    def __init__(self, bert_emb, cat_dict, labels):
        self.bert_emb = torch.tensor(bert_emb, dtype=torch.float32)
        self.cat_dict = {k: torch.tensor(v, dtype=torch.float32) for k, v in cat_dict.items()}
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.bert_emb[i], {k: v[i] for k, v in self.cat_dict.items()}, self.labels[i]

def collate_cat(batch):
    bert_embs = torch.stack([b[0] for b in batch])
    cat_names = batch[0][1].keys()
    cats = {name: torch.stack([b[1][name] for b in batch]) for name in cat_names}
    labels = torch.stack([b[2] for b in batch])
    return bert_embs, cats, labels

def make_cat_loader(bert_emb, cat_dict, labels, batch_size=64, shuffle=True):
    return DataLoader(CategoryDataset(bert_emb, cat_dict, labels), batch_size=batch_size, shuffle=shuffle, collate_fn=collate_cat)

def evaluate_gated(model, loader, threshold=0.5):
    model.eval(); all_labels, all_probs = [], []
    with torch.no_grad():
        for be, cats, lb in loader:
            be = be.to(DEVICE); cats = {k: v.to(DEVICE) for k, v in cats.items()}
            probs = torch.softmax(model(be, cats), dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy()); all_labels.extend(lb.numpy())
    all_labels, all_probs = np.array(all_labels), np.array(all_probs)
    return all_labels, (all_probs >= threshold).astype(int), all_probs

def calibrate_threshold(labels, probs, n_steps=199):
    best_t, best_f1 = 0.5, -1
    for t in np.linspace(0.01, 0.99, n_steps):
        f1 = f1_score(labels, (probs >= t).astype(int), zero_division=0)
        if f1 > best_f1: best_f1, best_t = f1, t
    return best_t, best_f1

def train_gated(train_loader, selection_loader, category_dims, seed, epochs=15, lr=1e-3, weight_decay=0.01, patience=5):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = AttentionGatedHybridDetector(category_dims).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = get_cosine_schedule_with_warmup(opt, int(0.1*len(train_loader)*epochs), len(train_loader)*epochs)
    criterion = nn.CrossEntropyLoss()
    best_val_f1, best_state, epochs_no_improve = -1, None, 0
    for ep in range(1, epochs+1):
        model.train()
        for be, cats, lb in train_loader:
            be = be.to(DEVICE); cats = {k: v.to(DEVICE) for k, v in cats.items()}; lb = lb.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(be, cats), lb)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
        sel_labels, sel_preds, _ = evaluate_gated(model, selection_loader)
        sel_f1 = f1_score(sel_labels, sel_preds, zero_division=0)
        if sel_f1 > best_val_f1:
            best_val_f1, best_state, epochs_no_improve = sel_f1, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience: break
    model.load_state_dict(best_state)
    sel_labels, _, sel_probs = evaluate_gated(model, selection_loader)
    calibrated_t, _ = calibrate_threshold(sel_labels, sel_probs)
    return model, best_val_f1, calibrated_t

def full_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {'accuracy': accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0), 'f1': f1_score(y_true, y_pred, zero_division=0),
            'fpr': fp/(fp+tn) if (fp+tn) > 0 else 0.0}


## 7. Train both variants — 5 seeds each, one test set (HC3 has no generator axis)

In [ ]:
train_loader = make_cat_loader(Xb['train'], Xcat['train'], y_train, batch_size=64, shuffle=True)
sel_loader_easy = make_cat_loader(Xb['val_easy'], Xcat['val_easy'], y_val_easy, batch_size=64, shuffle=False)
sel_loader_hard = make_cat_loader(Xb['val_hard'], Xcat['val_hard'], y_val_hard, batch_size=64, shuffle=False)
test_loader = make_cat_loader(Xb['test'], Xcat['test'], y_test, batch_size=128, shuffle=False)

def run_variant(variant_name, selection_loader, ckpt_prefix):
    results, raw, gate_log = [], [], []
    for seed in SEEDS:
        ckpt_name = f'{ckpt_prefix}_seed{seed}'
        if ckpt_exists(ckpt_name):
            seed_result = ckpt_load(ckpt_name)
        else:
            model, best_sel_f1, calibrated_t = train_gated(train_loader, selection_loader, category_dims, seed=seed)
            labels, preds, probs = evaluate_gated(model, test_loader, threshold=calibrated_t)
            m = full_metrics(labels, preds); m['seed'] = seed; m['threshold'] = calibrated_t
            with torch.no_grad():
                gates = []
                for be, cats, lb in test_loader:
                    be = be.to(DEVICE); cats = {k: v.to(DEVICE) for k, v in cats.items()}
                    _, gw = model(be, cats, return_gate=True); gates.append(gw.cpu().numpy())
                gate = dict(zip(category_dims.keys(), np.concatenate(gates).mean(axis=0).tolist()))
            seed_result = {'metrics': m, 'raw': (labels, preds), 'gate': gate, 'sel_f1': best_sel_f1, 'threshold': calibrated_t}
            ckpt_save(ckpt_name, seed_result)
            del model
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        results.append(seed_result['metrics'])
        raw.append(seed_result['raw'])
        gate_log.append(seed_result['gate'] | {'seed': seed})
        print(f'[{variant_name}] seed {seed}: sel_f1={seed_result["sel_f1"]:.4f}, threshold={seed_result["threshold"]:.3f}, '
              f'test_f1={seed_result["metrics"]["f1"]:.4f}')
    return results, raw, pd.DataFrame(gate_log)

print('=== Variant A: Easy (in-domain val) ===')
results_easy, raw_easy, gates_easy = run_variant('Easy', sel_loader_easy, 'hc3_hybrid_easy')

print()
print('=== Variant B: Hard (out-of-domain val) ===')
results_hard, raw_hard, gates_hard = run_variant('Hard', sel_loader_hard, 'hc3_hybrid_hard')


## 8. Compare — aggregate metrics, paired McNemar, gate weights, bootstrap CI

In [ ]:
def bootstrap_ci_f1(labels, preds, n_boot=3000, seed=42):
    rng = np.random.default_rng(seed); labels, preds = np.asarray(labels), np.asarray(preds); n = len(labels)
    stats = [f1_score(labels[idx], preds[idx], zero_division=0) for idx in (rng.integers(0,n,n) for _ in range(n_boot))]
    lo, hi = np.percentile(stats, [2.5, 97.5])
    return float(np.mean(stats)), float(lo), float(hi)

easy_df = pd.DataFrame(results_easy)
hard_df = pd.DataFrame(results_hard)
comparison = pd.DataFrame([
    {'variant': 'Easy (in-domain val)', 'f1_mean': easy_df.f1.mean(), 'f1_std': easy_df.f1.std(),
     'acc_mean': easy_df.accuracy.mean(), 'fpr_mean': easy_df.fpr.mean()},
    {'variant': 'Hard (out-of-domain val)', 'f1_mean': hard_df.f1.mean(), 'f1_std': hard_df.f1.std(),
     'acc_mean': hard_df.accuracy.mean(), 'fpr_mean': hard_df.fpr.mean()},
]).round(4)
print(comparison.to_string(index=False))

labels_easy0, preds_easy0 = raw_easy[0]
labels_hard0, preds_hard0 = raw_hard[0]
assert (labels_easy0 == labels_hard0).all()
a_ok, b_ok = (preds_hard0 == labels_easy0), (preds_easy0 == labels_easy0)
n01, n10 = int(np.sum(a_ok & ~b_ok)), int(np.sum(~a_ok & b_ok))
result = mcnemar([[0, n01], [n10, 0]], exact=True)
print()
print(f'McNemar (Hard vs Easy, seed 42): Hard_correct_Easy_wrong={n01}, Hard_wrong_Easy_correct={n10}, '
      f'p={result.pvalue:.6g}, winner={"Hard" if n01>n10 else ("Easy" if n10>n01 else "tie")}')

boot_rows = []
for variant_name, raw in [('Easy', raw_easy), ('Hard', raw_hard)]:
    for seed_i, (labels, preds) in enumerate(raw):
        mean_f1, lo, hi = bootstrap_ci_f1(labels, preds, seed=SEEDS[seed_i])
        boot_rows.append({'variant': variant_name, 'seed': SEEDS[seed_i], 'f1_mean': mean_f1, 'ci_lower': lo, 'ci_upper': hi})
boot_df = pd.DataFrame(boot_rows).round(4)
print()
print(boot_df.to_string(index=False))

print()
print('=== Gate weights: Easy selection ===')
print(gates_easy.to_string(index=False))
print()
print('=== Gate weights: Hard selection ===')
print(gates_hard.to_string(index=False))

comparison.to_csv(f'{ARTIFACTS_DIR}/HC3_easy_vs_hard_comparison.csv', index=False)
boot_df.to_csv(f'{ARTIFACTS_DIR}/HC3_bootstrap_ci.csv', index=False)
gates_easy.to_csv(f'{ARTIFACTS_DIR}/HC3_gates_easy.csv', index=False)
gates_hard.to_csv(f'{ARTIFACTS_DIR}/HC3_gates_hard.csv', index=False)
pd.DataFrame([{'n01': n01, 'n10': n10, 'p_value': result.pvalue}]).to_csv(f'{ARTIFACTS_DIR}/HC3_mcnemar.csv', index=False)

print()
print('If Hard significantly beats Easy here too (p<0.05, consistent F1 improvement), the M4')
print('finding generalizes to a second, independent dataset on the one axis of shift HC3 has --')
print('a much stronger claim than an M4-only result. If not, that is equally worth reporting:')
print('the effect may be M4-specific (e.g., tied to the generator axis specifically, which HC3 lacks).')
